# Part 6

In [1]:
import time
import folium
import pandas as pd
from IPython.display import display, clear_output
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, split, avg, count, window, to_timestamp
from pyspark.sql.types import LongType, DoubleType

HOST_IP      = '10.248.16.109'
KAFKA_BROKER = 'localhost:9092'
TOPIC        = 'journey.location'

existing = SparkSession.getActiveSession()
if existing:
    existing.stop()

spark = (SparkSession.builder.appName('Journey-Part7-Garmin')
    .master(f'spark://{HOST_IP}:7077')
    .config('spark.driver.host', HOST_IP)
    .config('spark.driver.bindAddress', HOST_IP)
    .config('spark.eventLog.enabled', 'true')
    .config('spark.eventLog.dir', 'file:///tmp/spark-events')
    .config('spark.sql.shuffle.partitions', '4')
    .getOrCreate())
spark


In [2]:
df_raw = (spark.readStream.format('kafka')
    .option('kafka.bootstrap.servers', KAFKA_BROKER)
    .option('subscribe', TOPIC)
    .option('startingOffsets', 'earliest')
    .load())

df_parsed = (df_raw
    .select(split(col('value').cast('string'), ',').alias('f'))
    .select(
        to_timestamp((col('f')[0].cast(LongType()) / 1000).cast(LongType())).alias('zeit'),
        col('f')[1].cast(DoubleType()).alias('lat'),
        col('f')[2].cast(DoubleType()).alias('lon'),
        col('f')[3].cast(DoubleType()).alias('alt'),
        col('f')[4].cast(DoubleType()).alias('hr'),
        col('f')[5].cast(DoubleType()).alias('temp'),
    )
    .filter(col('lat').between(-90, 90) & col('lon').between(-180, 180))
)


In [3]:
df_result = (df_parsed
    .withWatermark('zeit', '1 seconds')
    .groupBy(window(col('zeit'), '500 milliseconds'))
    .agg(
        avg('lat').alias('lat_mean'),
        avg('lon').alias('lon_mean'),
        avg('alt').alias('alt_mean'),
        avg('hr').alias('hr_mean'),
        avg('temp').alias('temp_mean'),
        count('lat').alias('n_samples'),
    )
    .select(
        col('window.start').alias('fenster_start'),
        'lat_mean', 'lon_mean', 'alt_mean', 'hr_mean', 'temp_mean', 'n_samples',
    ))


In [4]:
query = (df_result.writeStream
    .outputMode('complete')
    .format('memory')
    .queryName('garmin_track')
    .trigger(processingTime='1 seconds')
    .start())

print('Stream gestartet. Warte auf Daten vom Generator...')

Stream gestartet. Warte auf Daten vom Generator...


In [5]:
from IPython.display import IFrame

POLLS       = 500
REFRESH_SEC = 10
HTML_PATH   = '/home/bfh/rtdp/bfh-rtdp/journey/part6_garmin_live.html'

track_map   = None  
seen        = set()  
last_coord  = None  

for i in range(POLLS):
    time.sleep(REFRESH_SEC)

    pdf = (spark.sql('SELECT * FROM garmin_track ORDER BY fenster_start')
           .toPandas()
           .dropna(subset=['lat_mean', 'lon_mean']))

    clear_output(wait=True)

    if len(pdf) == 0:
        print(f'Poll {i+1}/{POLLS} – warte auf Daten vom Generator...')
        continue

    if track_map is None:
        track_map = folium.Map(
            location=[pdf['lat_mean'].iloc[0], pdf['lon_mean'].iloc[0]],
            zoom_start=15,
            control_scale=True,
        )

    new_rows = pdf[~pdf['fenster_start'].isin(seen)]

    if new_rows.empty:
        print(f'Poll {i+1}/{POLLS} – keine neuen Punkte, zeige letzte Karte')
        display(IFrame(src=HTML_PATH, width='100%', height='500px'))
        continue

    for _, row in new_rows.iterrows():
        coord = [row['lat_mean'], row['lon_mean']]

        if last_coord is not None:
            folium.PolyLine(
                [last_coord, coord],
                color='steelblue', weight=3, opacity=0.7,
            ).add_to(track_map)

        folium.Marker(
            location=coord,
            popup=folium.Popup(
                f"HR: {row['hr_mean']:.0f} bpm<br>"
                f"Alt: {row['alt_mean']:.0f} m<br>"
                f"Temp: {row['temp_mean']:.1f} °C<br>"
                f"Samples: {int(row['n_samples'])}",
                max_width=200,
            ),
            icon=folium.Icon(color='red', icon='heart', prefix='fa'),
        ).add_to(track_map)

        seen.add(row['fenster_start'])
        last_coord = coord

    track_map.save(HTML_PATH)
    display(IFrame(src=HTML_PATH, width='100%', height='500px'))
    print(f"Poll {i+1}/{POLLS} – {len(new_rows)} neue Punkte | gesamt {len(seen)} | "
          f"HR Ø {pdf['hr_mean'].mean():.0f} bpm | Alt Ø {pdf['alt_mean'].mean():.0f} m")


KeyboardInterrupt: 

In [ ]:
query.stop()
spark.stop()